<a href="https://colab.research.google.com/github/AnastasiyaPunko/Cygnus-Test-Pipeline/blob/main/WESstat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QC

In [ ]:
fastqc -t 4 -o /data/WES_260526/raw_data/L000/ *.fq.gz

In [ ]:
cutadapt -q 20,20 -a AGATCGGAAGAGCACACGTCTGAACTCCAGTCAC -A AGATCGGAAGAGCACACGTCTGAACTCCAGTCAC -a "A{10}" -A "A{10}" -e 0.15 -m 36 -o /data/WES_260526/qc/105_L000_R1_trimmed.fastq.gz -p /data/WES_260526/qc/105_L000_R2_trimmed.fastq.gz /data/WES_260526/raw_data/L000/105_L000_R1.fq.gz /data/WES_260526/raw_data/L000/105_L000_R2.fq.gz

In [ ]:
fastqc /data/WES_260526/qc/sample_L000_R1_trimmed.fastq.gz ; fastqc/data/WES_260526/qc/sample_L000_R2_trimmed.fastq.gz

# Alignment

In [ ]:
bwa mem -R '@RG\tID:GSS5-0798\tSM:wes\tPL:ILLUMINA' -t 3 /home/punko_a/genom/hg38.fa /data/WES_260526/qc/105_L000_R1_trimmed.fastq.gz /data/WES_260526/qc/105_L000_R2_trimmed.fastq.gz -o /data/WES_260526/align/105_trimmed.sam

In [ ]:
samtools view -Sb 2WES_trimmed.sam > 2WES_trimmed.bam

In [ ]:
samtools sort -m 3G -o sample_trimmed_sort.bam sample_trimmed.bam

In [ ]:
samtools index sample_trimmed_sort.bam

# The fraction of mapped sequence that is marked as duplicate

In [ ]:
picard MarkDuplicates \
      I=/data/WES_260526/align/sample_trimmed_sort.bam \
      O=/data/WES_260526/align/reportsMarkDuplicates/sample_trimmed_sort_marked_duplicates.bam \
      M=/data/WES_260526/align/reportsMarkDuplicates/sample_trimmed_sort_marked_dup_metrics.txt



*   2WES - 4.2%
*   1WES - 4%
*   111 - 4.9%
*   105 - 3.5%




# Total Reads (M)

In [ ]:
samtools view -c /data/WES_260526/align/sample_trimmed_sort.bam | awk '{print $1/1000000 " M"}'



*   2WES - 44.3967 M
*   1WES - 57.6013 M
*   111 - 62.4095 M
*   105 - 36.3857 M




In [ ]:
samtools stats /data/WES_260526/align/2WES_trimmed_sort.bam > /data/WES_260526/align/stats/2WES_quality_stats.txt

# Q>20 & Q>30

In [ ]:
# Расчет Q>20 и Q>30 из FFQ и LFQ
samtools stats /data/WES_260526/align/105_trimmed_sort.bam 2>/dev/null | grep -E "^(FFQ|LFQ)" | awk '
{
    for(i=2; i<=NF; i++) {
        qual = i-2
        count = $i
        if(qual > 20) q20 += count
        if(qual > 30) q30 += count
        total += count
    }
}
END {
    if(total > 0) {
        printf "Q>20: %.2f%%\n", q20/total*100
        printf "Q>30: %.2f%%\n", q30/total*100
        printf "Total bases: %d\n", total
    } else {
        print "Q>20: N/A"
        print "Q>30: N/A"
    }
}'

Q>20
*   2WES - 94.44%
*   1WES - 94.18%
*   111 - 94.57%
*   105 - 94.15%

Q>30
*   2WES - 90.52%
*   1WES - 90.12%
*   111 - 90.72%
*   105 - 89.97%

# Mapping reads %

See ***Sample_quality_stats.txt***

**raw total sequences** - total number of reads in a file, excluding supplementary and secondary reads. <br>
**reads mapped** - number of reads, paired or single, that are mapped. <br>
(reads mapped / raw total sequences) * 100

In [9]:
print((36290842 / 36310104) * 100 )

99.94695140504142


*   2WES - 99.94%
*   1WES - 99.93%
*   111 - 99.95%
*   105 - 99.95%






# Mean Target Coverage

In [ ]:
samtools depth -a -b /data/WES_260526/WES_V2.0.2_HG38_MT_COV.bed -Q 20 /data/WES_260526/align/2WES_trimmed_sort.bam | awk '{sum += $3; count++} END {printf "Mean Target Coverage: %.2fX\n", sum/count}'

*   2WES - 60.49x
*   1WES - 75.76x
*   111 - 86.72x
*   105 - 48.56x


# % On-Target